# Nonlinear MPC Motion Planner — from scratch

A **Nonlinear Model Predictive Control (NMPC)** motion planner for a
nonholonomic (unicycle) mobile robot navigating a cluttered 2-D arena, built
step by step with **CasADi** (symbolic optimization + IPOPT solver) and
visualized with **matplotlib**.

This notebook is self-contained and runs top-to-bottom in Google Colab.
Outline:

0. Install dependencies
1. Configuration (robot, limits, MPC weights, scenario)
2. The robot model — unicycle kinematics
3. Obstacles — static and time-varying boxes
4. The MPC controller — the optimal-control problem
5. The closed loop (receding horizon) + stuck-recovery
6. Visualization + run the two scenes
7. Inline animation
8. Experiments (answers to the review questions)

## Step 0 — Install dependencies

Colab ships with `numpy` and `matplotlib`; we only need **CasADi**.

In [ ]:
!pip -q install casadi

## Step 1 — Configuration

All the tunable constants in one place: the robot size, actuator limits, the
MPC horizon and cost weights, and the scenario (start pose + waypoints).

In [ ]:
import math
import numpy as np
import casadi

# --- robot & simulation ---
ROBOT_RADIUS_M = 0.4          # [m]  (a fairly large robot)
DT             = 0.1          # control period [s]
GOAL_THRESHOLD = 0.5          # [m]  "arrived" radius

# --- actuator limits ---
V_MIN, V_MAX   = 0.0, 3.0     # forward speed [m/s]
OMEGA_MAX      = 1.0          # yaw rate [rad/s]
OMEGA_MIN      = -OMEGA_MAX
A_MAX          = 1.5          # linear accel limit [m/s^2]
OMEGA_DOT_MAX  = 1.5          # yaw accel limit  [rad/s^2]

# --- MPC horizon & cost weights ---
N            = 30             # prediction horizon (3.0 s lookahead)
Q_pos        = 6.0           # running (stage) position cost
Q_term       = 60.0          # terminal position cost (dominant)
Q_theta      = 2.0           # terminal heading cost
R_v          = 0.02          # speed effort
R_omega      = 0.05          # yaw-rate effort
R_dv         = 0.3           # control-rate (smoothing) cost
W_SLACK      = 20000.0       # quadratic slack penalty
W_SLACK_LIN  = 8000.0        # linear (exact) slack penalty
SAFE_MARGIN  = 0.20          # clearance beyond the robot radius [m]
K_NEAR       = 10            # constrain only the K nearest obstacles

# --- scenario (start + waypoints, each [x, y, theta]) ---
START_POSE = (1.0, 1.0, 0.0)
WAYPOINTS = [
    (5.0,  2.0, 1.57),
    (10.0, 7.0, 1.57),
    (5.0,  12.0, 3.14),
    (13.0, 13.0, 0.0),
]
XLIM, YLIM = (-1, 15), (-1, 15)
print("config loaded")

## Step 2 — The robot model (unicycle kinematics)

The robot is a **unicycle** (differential-drive) with state
$x = [p_x,\, p_y,\, \theta]$ and control $u = [v,\, \omega]$. The governing
motion equation, discretized with forward Euler at step $\Delta t$, is:

$$
\begin{aligned}
p_{x,k+1} &= p_{x,k} + v_k \cos\theta_k\,\Delta t \\
p_{y,k+1} &= p_{y,k} + v_k \sin\theta_k\,\Delta t \\
\theta_{k+1} &= \theta_k + \omega_k\,\Delta t
\end{aligned}
$$

This same equation is used both to *simulate* the true robot and, inside the
MPC, as the dynamics constraint the plan must obey.

In [ ]:
def unicycle_step(state, control, dt):
    x, y, th = state
    v, om = control
    return [x + v*math.cos(th)*dt,
            y + v*math.sin(th)*dt,
            th + om*dt]

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

class Robot:
    def __init__(self, x, y, theta):
        self.x, self.y, self.theta = x, y, theta
        self.r = ROBOT_RADIUS_M
        self.collided = False

    @property
    def state(self):
        return [self.x, self.y, self.theta]

    def apply(self, v, omega, dt):
        v = clamp(v, V_MIN, V_MAX)
        omega = clamp(omega, OMEGA_MIN, OMEGA_MAX)
        self.x, self.y, self.theta = unicycle_step(self.state, [v, omega], dt)

    def check_collision(self, obstacles, t=0.0):
        self.collided = any(o.collides_circle(self.x, self.y, self.r, t)
                            for o in obstacles)
        return self.collided

print("Robot ready")

## Step 3 — Obstacles

Obstacles are **oriented rectangles** that may translate (along their initial
heading at speed $v$) and rotate (rate $\omega$). Their pose at time $t$ is

$$
c(t) = c^0 + v\,t\,[\cos\alpha^0,\ \sin\alpha^0]^\top,\qquad
\alpha(t) = \alpha^0 + \omega\,t.
$$

Static obstacles simply have $v=\omega=0$.

We measure proximity with a **signed-distance function** to the (rotated) box:
it is positive outside, negative inside. This is smooth enough to differentiate,
which is what lets us put it inside the optimizer as a constraint.

The obstacle sets below are the two scenes from the project (**static**: 13
fixed boxes including a 4 m diagonal wall; **dynamic**: 20 moving/rotating
boxes), embedded directly so the notebook is self-contained.

In [ ]:
class Obstacle:
    '''Rotated rectangle; may translate along its initial heading and spin.'''
    def __init__(self, cx, cy, w, h, angle=0.0, v=0.0, omega=0.0):
        self.cx0, self.cy0 = cx, cy
        self.w, self.h = w, h
        self.angle0 = angle
        self.v, self.omega = v, omega
        self._dx = v*math.cos(angle)   # constant drift (fixed initial heading)
        self._dy = v*math.sin(angle)

    @property
    def is_dynamic(self):
        return self.v != 0.0 or self.omega != 0.0

    def pose_at(self, t):
        return (self.cx0 + self._dx*t, self.cy0 + self._dy*t,
                self.angle0 + self.omega*t)

    def corners(self, t=0.0):
        cx, cy, ang = self.pose_at(t)
        hw, hh = self.w/2, self.h/2
        ca, sa = math.cos(ang), math.sin(ang)
        pts = []
        for lx, ly in [(-hw,-hh),(hw,-hh),(hw,hh),(-hw,hh)]:
            pts.append((cx + lx*ca - ly*sa, cy + lx*sa + ly*ca))
        return pts

    def signed_distance(self, px, py, m, t=0.0):
        '''m is the math backend: `math` (scalars) or `casadi` (symbolic).'''
        cx = self.cx0 + self._dx*t
        cy = self.cy0 + self._dy*t
        ang = self.angle0 + self.omega*t
        dx, dy = px - cx, py - cy
        c, s = m.cos(-ang), m.sin(-ang)
        lx = dx*c - dy*s
        ly = dx*s + dy*c
        ax = m.fabs(lx) - self.w/2
        ay = m.fabs(ly) - self.h/2
        if m is math:
            outside = math.hypot(max(ax,0.0), max(ay,0.0))
            inside = min(max(ax,ay), 0.0)
        else:
            outside = m.sqrt(m.fmax(ax,0)**2 + m.fmax(ay,0)**2 + 1e-6)
            inside = m.fmin(m.fmax(ax,ay), 0.0)
        return outside + inside

    def collides_circle(self, px, py, r=ROBOT_RADIUS_M, t=0.0):
        return self.signed_distance(px, py, math, t) < r


# (cx, cy, w, h, angle, v, omega)
STATIC_OBS = [
    (3.0,3.2,1.6,0.8,0.0,0.0,0.0), (3.5,0.3,1.0,0.5,0.3,0.0,0.0),
    (7.8,10.0,1.6,0.8,-0.4,0.0,0.0), (9.0,8.5,1.2,0.6,0.2,0.0,0.0),
    (6.0,9.0,1.0,0.5,0.0,0.0,0.0), (8.5,13.0,1.8,0.9,0.0,0.0,0.0),
    (10.5,11.0,1.4,0.7,0.4,0.0,0.0), (7.0,11.5,1.0,0.5,-0.2,0.0,0.0),
    (4.5,7.0,1.4,0.7,0.7,0.0,0.0), (12.0,8.0,1.2,0.6,-0.5,0.0,0.0),
    (1.5,6.0,1.2,0.6,0.1,0.0,0.0), (7.0,4.0,4.0,0.6,0.8,0.0,0.0),
    (8.5,6.0,2.0,0.5,0.0,0.0,0.0),
]
DYNAMIC_OBS = [
    (3.0,3.2,1.6,0.8,0.0,0.05,0.012), (3.5,0.3,1.0,0.5,1.2,0.08,-0.015),
    (0.5,4.0,0.8,0.4,0.8,0.14,0.0), (7.2,4.0,1.6,0.8,1.8,0.04,0.01),
    (8.5,6.5,1.4,0.7,-0.5,0.06,-0.012), (6.2,2.5,1.0,0.5,0.5,0.0,0.18),
    (6.0,5.5,0.8,0.4,1.0,0.14,-0.025), (7.8,10.0,1.4,0.7,-0.3,0.05,0.012),
    (9.0,8.5,1.2,0.6,2.2,0.08,-0.015), (6.0,9.0,1.0,0.5,1.57,0.06,0.0),
    (10.0,9.0,0.7,0.35,0.0,0.16,0.035), (8.5,13.0,1.6,0.8,3.14,0.035,0.008),
    (10.5,11.0,1.4,0.7,-0.8,0.055,-0.012), (7.0,11.5,1.0,0.5,0.5,0.07,0.015),
    (11.0,14.0,0.8,0.4,-1.57,0.12,0.0), (4.5,7.0,1.4,0.7,0.7,0.05,-0.012),
    (12.0,8.0,1.2,0.6,2.0,0.06,0.015), (1.5,6.0,1.2,0.6,0.1,0.04,-0.01),
    (12.0,4.0,1.0,0.5,1.0,0.08,0.02), (3.5,10.5,0.7,0.35,-0.5,0.18,-0.035),
]

def load_scene(scene):
    data = STATIC_OBS if scene == "static" else DYNAMIC_OBS
    return [Obstacle(*row) for row in data]

print("static:", len(load_scene("static")), "obstacles |",
      "dynamic:", len(load_scene("dynamic")), "obstacles")

## Step 4 — The MPC controller (the optimal control problem)

At every control step we solve the following optimization over the next
$N$ steps, then apply only the first control and re-solve at the next step
(**receding horizon**):

$$
\min_{X,\,U,\,S}\;\;
\sum_{k=0}^{N} Q_p\lVert p_k - p_g\rVert^2
+ Q_N\lVert p_N - p_g\rVert^2
+ Q_\theta\big(1 - \cos(\theta_N - \theta_g)\big)
+ \sum_{k=0}^{N-1}\!\big(R_v v_k^2 + R_\omega \omega_k^2\big)
+ \sum_{k=0}^{N-2}\! R_\Delta\lVert u_{k+1}-u_k\rVert^2
+ \sum_{i,k}\!\big(W_2 S_{i,k}^2 + W_1 S_{i,k}\big)
$$

subject to, for $k = 0,\dots,N-1$:

$$
\begin{aligned}
&\text{(dynamics)} && x_{k+1} = f(x_k, u_k)\ \ \text{(the unicycle model)}\\
&\text{(initial state)} && x_0 = x_\text{meas}\\
&\text{(actuator bounds)} && 0 \le v_k \le 3,\quad -1 \le \omega_k \le 1\\
&\text{(rate limits)} && |v_{k+1}-v_k|\le a_{\max}\Delta t,\quad |\omega_{k+1}-\omega_k|\le \dot\omega_{\max}\Delta t\\
&\text{(collision avoidance)} && d_i(p_k,\, t_0+k\Delta t) \ge (r_\text{robot}+m) - S_{i,k},\quad S_{i,k}\ge 0
\end{aligned}
$$

**Key ideas:**
- The **terminal cost** $Q_N$ dominates so the robot is pulled to the goal
  (and doesn't settle for "stay put").
- The collision constraint is evaluated at each obstacle's **predicted future
  pose** $t_0+k\Delta t$ — genuine time-varying avoidance for moving obstacles.
- The constraint is **soft** (slack $S \ge 0$) with an **exact penalty**:
  a quadratic term $W_2$ *plus a large linear term* $W_1$. The linear term makes
  any intrusion first-order costly, so a soft constraint behaves like a hard one
  (no corner-cutting) while the problem always stays feasible.
- Only the **$K$ nearest obstacles** (selected along the predicted path) are
  constrained each solve, which keeps the problem small and fast.

In [ ]:
_FAR_SLOT = [1e4, 1e4, 0.01, 0.01, 0.0, 0.0, 0.0, 0.0]

def _obs_params(o):
    return [o.cx0, o.cy0, o.w/2, o.h/2, o.angle0, o._dx, o._dy, o.omega]

def _sd_param(px, py, p, t):
    '''Signed distance to a *parameterized* obstacle (its 8 params in p).'''
    cx = p[0] + p[5]*t
    cy = p[1] + p[6]*t
    ang = p[4] + p[7]*t
    dx, dy = px - cx, py - cy
    c, s = casadi.cos(-ang), casadi.sin(-ang)
    lx = dx*c - dy*s
    ly = dx*s + dy*c
    ax = casadi.fabs(lx) - p[2]
    ay = casadi.fabs(ly) - p[3]
    outside = casadi.sqrt(casadi.fmax(ax,0)**2 + casadi.fmax(ay,0)**2 + 1e-6)
    inside = casadi.fmin(casadi.fmax(ax,ay), 0.0)
    return outside + inside


class MPCController:
    def __init__(self, goal, obstacles):
        self.obstacles = obstacles
        opti = casadi.Opti()
        X = opti.variable(3, N+1)          # states
        U = opti.variable(2, N)            # controls
        P_start = opti.parameter(3)
        P_goal  = opti.parameter(3)
        P_t0    = opti.parameter(1)

        K = min(K_NEAR, len(obstacles))
        self.K = K
        P_obs = opti.parameter(8, K)       # nearest-K obstacle params
        S = opti.variable(K, N+1)          # slack

        # dynamics constraints (the unicycle model)
        for k in range(N):
            x, y, th = X[0,k], X[1,k], X[2,k]
            v, om = U[0,k], U[1,k]
            opti.subject_to(X[0,k+1] == x + v*casadi.cos(th)*DT)
            opti.subject_to(X[1,k+1] == y + v*casadi.sin(th)*DT)
            opti.subject_to(X[2,k+1] == th + om*DT)

        opti.subject_to(X[:,0] == P_start)                       # initial state
        opti.subject_to(opti.bounded(V_MIN, U[0,:], V_MAX))      # speed bounds
        opti.subject_to(opti.bounded(OMEGA_MIN, U[1,:], OMEGA_MAX))

        for k in range(N-1):                                     # rate limits
            dv  = U[0,k+1] - U[0,k]
            dom = U[1,k+1] - U[1,k]
            opti.subject_to(opti.bounded(-A_MAX*DT, dv, A_MAX*DT))
            opti.subject_to(opti.bounded(-OMEGA_DOT_MAX*DT, dom, OMEGA_DOT_MAX*DT))

        safe = ROBOT_RADIUS_M + SAFE_MARGIN                      # collision avoid
        for i in range(K):
            for k in range(N+1):
                t = P_t0 + k*DT
                d = _sd_param(X[0,k], X[1,k], P_obs[:,i], t)
                opti.subject_to(d >= safe - S[i,k])
                opti.subject_to(S[i,k] >= 0)

        # ---- cost ----
        cost = 0
        for k in range(N+1):
            cost += Q_pos*((X[0,k]-P_goal[0])**2 + (X[1,k]-P_goal[1])**2)
        cost += Q_term*((X[0,N]-P_goal[0])**2 + (X[1,N]-P_goal[1])**2)
        cost += Q_theta*(1 - casadi.cos(X[2,N]-P_goal[2]))
        for k in range(N):
            cost += R_v*U[0,k]**2 + R_omega*U[1,k]**2
        for k in range(N-1):
            cost += R_dv*((U[0,k+1]-U[0,k])**2 + (U[1,k+1]-U[1,k])**2)
        for i in range(K):
            for k in range(N+1):
                cost += W_SLACK*S[i,k]**2 + W_SLACK_LIN*S[i,k]
        opti.minimize(cost)

        opti.solver("ipopt", {"print_time": False, "expand": True},
                    {"print_level": 0, "sb": "yes", "max_iter": 200,
                     "acceptable_tol": 1e-3, "acceptable_iter": 5,
                     "warm_start_init_point": "yes", "mu_strategy": "adaptive"})

        self.opti, self.X, self.U, self.S = opti, X, U, S
        self.P_start, self.P_goal, self.P_t0, self.P_obs = P_start, P_goal, P_t0, P_obs
        self._obs_p = [_obs_params(o) for o in obstacles]
        self.opti.set_value(self.P_goal, self._goal3(goal))
        self.opti.set_value(self.P_t0, 0.0)
        self._select_obstacles(np.array(self._goal3(goal))[:2], 0.0)
        self.last_X = self.last_U = None

    @staticmethod
    def _goal3(goal):
        if goal is None: return [0.0,0.0,0.0]
        if len(goal) == 2: return [goal[0], goal[1], 0.0]
        return [goal[0], goal[1], goal[2]]

    def _select_obstacles(self, pos, t0, path=None):
        n = len(self.obstacles)
        if n == 0:
            self.opti.set_value(self.P_obs, np.array([_FAR_SLOT]*self.K).T)
            return
        if path is not None:
            pts = [(path[0,k], path[1,k], t0+k*DT) for k in range(path.shape[1])]
        else:
            pts = [(pos[0], pos[1], t0)]
        def key(i):
            o = self.obstacles[i]; best = float("inf")
            for px, py, t in pts:
                cx = o.cx0 + o._dx*t; cy = o.cy0 + o._dy*t
                best = min(best, (cx-px)**2 + (cy-py)**2)
            return best
        order = sorted(range(n), key=key)
        cols = [self._obs_p[i] for i in order[:self.K]]
        while len(cols) < self.K: cols.append(_FAR_SLOT)
        self.opti.set_value(self.P_obs, np.array(cols).T)

    def solve(self, current_state, goal=None, t0=0.0):
        self.opti.set_value(self.P_start, current_state)
        self.opti.set_value(self.P_t0, t0)
        if goal is not None:
            self.opti.set_value(self.P_goal, self._goal3(goal))
        self._select_obstacles(np.asarray(current_state)[:2], t0, path=self.last_X)
        if self.last_X is not None:                              # warm start
            self.opti.set_initial(self.X, np.hstack([self.last_X[:,1:], self.last_X[:,-1:]]))
            self.opti.set_initial(self.U, np.hstack([self.last_U[:,1:], self.last_U[:,-1:]]))
        try:
            sol = self.opti.solve()
            self.last_X = sol.value(self.X); self.last_U = sol.value(self.U)
            return self.last_U[0,0], self.last_U[1,0], self.last_X
        except RuntimeError:                                     # graceful fallback
            if self.last_U is not None:
                return self.last_U[0,0], self.last_U[1,0], self.last_X
            return 0.0, 0.0, None

print("MPCController ready")

## Step 5 — The closed loop (receding horizon) + stuck-recovery

Each tick we solve the MPC for the current waypoint, apply $u_0$, step the robot,
advance the sim clock (which moves the dynamic obstacles), and log everything.
When the robot gets within `GOAL_THRESHOLD` of a waypoint we advance to the next.

A pure local MPC can get stuck in **local minima** (e.g. a box on the straight
line to the goal, where any escape briefly increases cost). We add a small
**stuck-recovery** behavior — the same idea as Nav2's recovery behaviors: after
~1.5 s of near-zero speed, rotate toward the goal bearing and creep forward
(only if the next step is collision-free), then hand control back to the MPC
with a fresh heading so it can find a descent direction.

`refresh` is an optional hook used later to swap the obstacle-prediction model.

In [ ]:
def _min_clearance(x, y, obstacles, t):
    if not obstacles: return float("inf")
    return min(o.signed_distance(x, y, math, t) for o in obstacles)

def closed_loop(controller, sim_obstacles, waypoints, start_pose,
                use_recovery=True, refresh=None, max_steps=2000):
    robot = Robot(*start_pose)
    sim_t = 0.0; wp = 0
    log = {k: [] for k in ("traj","ctrl","clr","solve_ms","pred","t","recov")}
    stuck = recov = 0
    STUCK_SPEED, STUCK_STEPS, RECOVERY_STEPS = 0.05, 15, 12
    collisions = 0; reached = set()

    import time
    for step in range(max_steps):
        target = waypoints[wp]
        dist = math.hypot(robot.x-target[0], robot.y-target[1])

        if refresh is not None:
            refresh(controller, sim_obstacles, sim_t)

        if recov > 0:
            bearing = math.atan2(target[1]-robot.y, target[0]-robot.x)
            ang_err = math.atan2(math.sin(bearing-robot.theta),
                                 math.cos(bearing-robot.theta))
            omega = clamp(2.0*ang_err, -OMEGA_MAX, OMEGA_MAX)
            v = 0.0
            if abs(ang_err) < 0.5:
                nx = robot.x + 0.3*math.cos(robot.theta)*DT
                ny = robot.y + 0.3*math.sin(robot.theta)*DT
                if all(not o.collides_circle(nx, ny, robot.r+0.1, sim_t)
                       for o in sim_obstacles):
                    v = 0.3
            X_pred = None; recov -= 1; solve_ms = 0.0; in_recov = True
        else:
            t0 = time.time()
            v, omega, X_pred = controller.solve(robot.state, target, t0=sim_t)
            solve_ms = (time.time()-t0)*1000.0; in_recov = False

        robot.apply(v, omega, DT); sim_t += DT
        if robot.check_collision(sim_obstacles, sim_t): collisions += 1

        log["traj"].append((robot.x, robot.y)); log["ctrl"].append((v, omega))
        log["clr"].append(_min_clearance(robot.x, robot.y, sim_obstacles, sim_t))
        log["solve_ms"].append(solve_ms); log["t"].append(sim_t)
        log["pred"].append(None if X_pred is None else np.array(X_pred))
        log["recov"].append(in_recov)

        if use_recovery and not in_recov:
            if abs(v) < STUCK_SPEED and dist > GOAL_THRESHOLD:
                stuck += 1
                if stuck >= STUCK_STEPS: recov = RECOVERY_STEPS; stuck = 0
            else:
                stuck = 0

        if dist < GOAL_THRESHOLD:
            reached.add(wp)
            if wp < len(waypoints)-1: wp += 1
            else: break

    for k in ("traj","ctrl","clr","solve_ms","t"):
        log[k] = np.array(log[k])
    log["collisions"] = collisions
    log["reached"] = len(reached); log["n_wp"] = len(waypoints)
    log["steps"] = step+1
    return log

print("closed_loop ready")

## Step 6 — Visualization and running the two scenes

We plot the trajectory over the obstacle field, the control inputs, and the
solver time per step. (Dynamic obstacles are drawn at their *final* pose.)

In [ ]:
import matplotlib.pyplot as plt

def draw_obstacles(ax, obstacles, t=0.0):
    for o in obstacles:
        cs = o.corners(t) + [o.corners(t)[0]]
        xs = [c[0] for c in cs]; ys = [c[1] for c in cs]
        fill = "navajowhite" if o.is_dynamic else "lightcoral"
        edge = "darkorange" if o.is_dynamic else "indianred"
        ax.fill(xs, ys, color=fill, alpha=0.7, zorder=1)
        ax.plot(xs, ys, color=edge, lw=1, zorder=1)

def plot_result(log, obstacles, scene, t_draw=None):
    traj = log["traj"]; ctrl = log["ctrl"]
    st = log["solve_ms"]; st = st[st > 0]           # ignore recovery steps
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(16, 5))
    draw_obstacles(a1, obstacles, t=(log["t"][-1] if t_draw is None else t_draw))
    a1.plot(traj[:,0], traj[:,1], "b-", lw=2, label="trajectory")
    a1.plot(START_POSE[0], START_POSE[1], "ko", ms=9, label="start")
    for wp in WAYPOINTS: a1.plot(wp[0], wp[1], "rx", ms=10, mew=2)
    a1.set_title(f"Trajectory ({scene}) — reached {log['reached']}/{log['n_wp']}, "
                 f"collisions={log['collisions']}")
    a1.set_xlim(XLIM); a1.set_ylim(YLIM); a1.set_aspect("equal")
    a1.set_xlabel("x [m]"); a1.set_ylabel("y [m]"); a1.grid(alpha=.3); a1.legend()

    tt = np.arange(len(ctrl))*DT
    a2.plot(tt, ctrl[:,0], "b-", label="v [m/s]")
    a2.plot(tt, ctrl[:,1], "r-", label="omega [rad/s]")
    a2.axhline(V_MAX, color="b", ls="--", alpha=.4)
    a2.axhline(OMEGA_MAX, color="r", ls="--", alpha=.4)
    a2.axhline(-OMEGA_MAX, color="r", ls="--", alpha=.4)
    a2.set_title("Control inputs"); a2.set_xlabel("time [s]"); a2.grid(alpha=.3); a2.legend()

    a3.plot(st, "k-", lw=0.8); a3.fill_between(range(len(st)), st, alpha=.3)
    if len(st): a3.axhline(st.mean(), color="r", ls="--",
                           label=f"mean {st.mean():.0f} ms")
    a3.set_title("Solver time / step"); a3.set_xlabel("MPC step")
    a3.set_ylabel("ms"); a3.grid(alpha=.3); a3.legend()
    plt.tight_layout(); plt.show()

print("plotting ready")

In [ ]:
# --- run the STATIC scene ---
obs_static = load_scene("static")
ctrl = MPCController(WAYPOINTS[0], obs_static)
log_static = closed_loop(ctrl, obs_static, WAYPOINTS, START_POSE)
print(f"static: reached {log_static['reached']}/{log_static['n_wp']} waypoints, "
      f"collisions={log_static['collisions']}, steps={log_static['steps']}")
plot_result(log_static, obs_static, "static")

In [ ]:
# --- run the DYNAMIC scene (20 moving/rotating obstacles) ---
obs_dyn = load_scene("dynamic")
ctrl = MPCController(WAYPOINTS[0], obs_dyn)
log_dyn = closed_loop(ctrl, obs_dyn, WAYPOINTS, START_POSE)
print(f"dynamic: reached {log_dyn['reached']}/{log_dyn['n_wp']} waypoints, "
      f"collisions={log_dyn['collisions']}, steps={log_dyn['steps']}")
plot_result(log_dyn, obs_dyn, "dynamic")

## Step 7 — Inline animation

An animation of the closed loop, rendered as an HTML5/JS video that plays
directly in Colab. Green dots are the MPC's predicted horizon at each frame;
orange/red boxes are the obstacles (which move in the dynamic scene).

In [ ]:
from matplotlib import animation
from IPython.display import HTML

def animate_run(log, obstacles, scene, stride=2):
    idx = list(range(0, log["steps"], stride))
    traj = log["traj"]
    fig, ax = plt.subplots(figsize=(6.5, 6.5))

    def frame(fi):
        ax.clear()                                   # robust: redraw each frame
        ax.set_xlim(XLIM); ax.set_ylim(YLIM); ax.set_aspect("equal")
        ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
        i = idx[fi]; t = log["t"][i]
        ax.set_title(f"MPC closed loop — {scene}   (t = {t:4.1f} s)")
        draw_obstacles(ax, obstacles, t=t)
        for wp in WAYPOINTS:
            ax.plot(wp[0], wp[1], "rx", ms=9, mew=2)
        ax.plot(traj[:i+1, 0], traj[:i+1, 1], "b-", lw=2)     # path so far
        pred = log["pred"][i]
        if pred is not None:                                  # predicted horizon
            ax.plot(pred[0, :], pred[1, :], "g.-", ms=3, lw=1)
        ax.plot(traj[i, 0], traj[i, 1], "o", color="royalblue", ms=10)  # robot
        return []

    anim = animation.FuncAnimation(fig, frame, frames=len(idx),
                                   interval=60, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

# animate the static scene (change to log_dyn / obs_dyn for the moving one)
animate_run(log_static, obs_static, "static")

## Step 8 — Experiments (answers to the review questions)

Three focused experiments that probe the planner's behavior and limits.

### Q3 — Behavior with the obstacle-avoidance term removed

We give the controller only a single far-away dummy obstacle, so the collision
constraint is present but trivially satisfied everywhere. The problem reduces to
**pure point-to-point pose regulation**: smooth arcs between waypoints, speed
saturating at $v_\max$, and (as expected) driving straight through the obstacles
it is no longer told about. This isolates and verifies the tracking layer.

In [ ]:
obs_static = load_scene("static")
noavoid = [Obstacle(1e4, 1e4, 0.01, 0.01)]      # single far dummy => no avoidance
ctrl = MPCController(WAYPOINTS[0], noavoid)
log_q3 = closed_loop(ctrl, obs_static, WAYPOINTS, START_POSE, use_recovery=False)
print(f"reached {log_q3['reached']}/{log_q3['n_wp']} waypoints, "
      f"collisions vs real boxes = {log_q3['collisions']}")
plot_result(log_q3, obs_static, "no obstacle avoidance")

### Q4a — Long obstacles → local minima

Same start and goal, with a thin wall across the corridor made progressively
longer. A short wall is routed around; once the wall exceeds the horizon's
lateral reach ($N=30$, $v_\max=3$ → about 9 m), the robot noses up to the wall
and **stalls**, because every direction toward the goal locally increases cost.
This is the classic local-minimum limitation of purely local planning.

In [ ]:
start = (7.0, 1.0, math.pi/2); goal = (7.0, 13.0, math.pi/2)
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
for ax, L in zip(axes, [2.0, 5.0, 9.0]):
    wall = [Obstacle(7.0, 7.0, L, 0.5, angle=0.0)]
    ctrl = MPCController(goal, wall)
    log = closed_loop(ctrl, wall, [goal], start, use_recovery=False, max_steps=400)
    draw_obstacles(ax, wall)
    tr = log["traj"]
    ax.plot(tr[:,0], tr[:,1], "b-", lw=2)
    ax.plot(*start[:2], "ko", ms=8); ax.plot(goal[0], goal[1], "rx", ms=11, mew=2.5)
    ax.plot(tr[-1,0], tr[-1,1], "b*", ms=15)
    fd = math.hypot(tr[-1,0]-goal[0], tr[-1,1]-goal[1])
    verdict = "REACHED" if log["reached"] else f"STUCK ({fd:.1f} m short)"
    ax.set_title(f"wall = {L:.0f} m\n{verdict}")
    ax.set_xlim(0,14); ax.set_ylim(0,14); ax.set_aspect("equal"); ax.grid(alpha=.3)
plt.suptitle("Long-obstacle failure onset (recovery off)"); plt.tight_layout(); plt.show()

### Q4b — Fast dynamic obstacles → it's about prediction, not speed

A box crosses the robot's path at increasing speed (up to the robot's own speed).
We compare two obstacle-prediction models in the controller:

- **exact**: propagate the obstacle's pose forward using its known velocity
  (what the delivered controller does) → clears the obstacle at **every** speed.
- **static**: re-observe the obstacle each tick but assume it is frozen over the
  horizon (the common case without velocity estimates) → **collides** once the
  obstacle moves at half the ego speed or faster.

So the failure mode is driven by **prediction error**, not obstacle speed itself.

In [ ]:
def crossing_scene(v_obs):
    return [Obstacle(7.0, 7.0 - v_obs*2.0, 1.4, 1.0, angle=math.pi/2, v=v_obs)]

def make_refresh(use_velocity):
    def refresh(controller, obstacles, t):
        for i, o in enumerate(obstacles):
            if use_velocity:
                controller._obs_p[i] = [o.cx0, o.cy0, o.w/2, o.h/2, o.angle0,
                                        o._dx, o._dy, o.omega]
            else:
                cx, cy, ang = o.pose_at(t)
                controller._obs_p[i] = [cx, cy, o.w/2, o.h/2, ang, 0.0, 0.0, 0.0]
    return refresh

start = (1.0, 7.0, 0.0); goal = (14.0, 7.0, 0.0)
for use_velocity, name in [(True, "exact (delivered controller)"),
                           (False, "static assumption (baseline)")]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, spd in zip(axes, [0.3, 1.5, 3.0]):
        obs = crossing_scene(spd)
        stand_ins = [Obstacle(o.cx0, o.cy0, o.w, o.h, o.angle0, 0, 0) for o in obs]
        ctrl = MPCController(goal, stand_ins)
        log = closed_loop(ctrl, obs, [goal], start, use_recovery=False,
                          refresh=make_refresh(use_velocity), max_steps=800)
        tend = log["t"][-1]
        for tt in (0.0, tend*0.4, tend*0.7, tend): draw_obstacles(ax, obs, t=tt)
        tr = log["traj"]; ax.plot(tr[:,0], tr[:,1], "b-", lw=2)
        ax.plot(*start[:2], "ko", ms=7); ax.plot(goal[0], goal[1], "rx", ms=10, mew=2)
        mc = log["clr"].min() - ROBOT_RADIUS_M
        tag = "COLLISION" if log["collisions"] else "cleared"
        ax.set_title(f"v_obs={spd} m/s\n{tag}, min clr={mc:+.2f} m")
        ax.set_xlim(-1,15); ax.set_ylim(-1,15); ax.set_aspect("equal"); ax.grid(alpha=.3)
    plt.suptitle(f"Q4b — predictor: {name}"); plt.tight_layout(); plt.show()

---
### Summary

- The MPC **regulates directly to goal poses** (waypoints) — there is no global
  planner and no reference trajectory being tracked.
- Removing the collision term (**Q3**) confirms the tracking layer is clean; all
  the difficulty is in the avoidance constraints.
- **Long obstacles (Q4a)** trap a purely local MPC in a local minimum — a
  fundamental limitation, mitigated on the delivered map by a recovery behavior.
- For **fast dynamic obstacles (Q4b)**, the delivered controller clears every
  speed because it predicts each obstacle's future pose; the failure mode
  appears only under an imperfect (static) prediction model.